# T12 — Finalizacija pregledanih SFT primjera

Ovaj notebook koristi već pregledani `data/processed/sft_review.csv` i finalizuje tačno 600 odobrenih primjera.

Važno:
- ne remapira `final_category` ponovo;
- čuva punu `messages_json` strukturu za multi-turn primjere;
- provjerava `source_ids` protiv `data/sources.csv`;
- izvozi `data/processed/all_examples.jsonl` i `artifacts/reports/sft_review_summary.csv`.


In [1]:
from pathlib import Path
import json
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Nije pronađen korijen projekta.')

PROJECT_ROOT = find_project_root()
REVIEW_CSV_PATH = PROJECT_ROOT / 'data' / 'processed' / 'sft_review.csv'
FINAL_PATH = PROJECT_ROOT / 'data' / 'processed' / 'all_examples.jsonl'
REPORT_PATH = PROJECT_ROOT / 'artifacts' / 'reports' / 'sft_review_summary.csv'
SOURCES_PATH = PROJECT_ROOT / 'data' / 'sources.csv'

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Review CSV:', REVIEW_CSV_PATH, REVIEW_CSV_PATH.exists())
print('Sources CSV:', SOURCES_PATH, SOURCES_PATH.exists())


PROJECT_ROOT: /home/sinisa/Desktop/Sve/Fakultet/CETVRTA GODINA/DRUGI SEMESTAR/Vjestacka inteligencija/PROJEKAT/bih-tourist-chatbot
Review CSV: /home/sinisa/Desktop/Sve/Fakultet/CETVRTA GODINA/DRUGI SEMESTAR/Vjestacka inteligencija/PROJEKAT/bih-tourist-chatbot/data/processed/sft_review.csv True
Sources CSV: /home/sinisa/Desktop/Sve/Fakultet/CETVRTA GODINA/DRUGI SEMESTAR/Vjestacka inteligencija/PROJEKAT/bih-tourist-chatbot/data/sources.csv True


In [2]:
# Učitaj već završeni review CSV. Ne resetuj ga i ne remapiraj final_category.
assert REVIEW_CSV_PATH.exists(), f'Nedostaje: {REVIEW_CSV_PATH}'
review_df = pd.read_csv(REVIEW_CSV_PATH, keep_default_na=False)

required_cols = [
    'example_id', 'type', 'destination_ids', 'source_ids',
    'review_status', 'final_category', 'reviewed', 'messages_json'
]
missing = [c for c in required_cols if c not in review_df.columns]
assert not missing, f'Nedostaju kolone u finalnom review CSV-u: {missing}'

print('Broj redova:', len(review_df))
display(review_df['review_status'].value_counts(dropna=False).rename_axis('review_status').reset_index(name='count'))


Broj redova: 926


,review_status,count
0,approved,600
1,reject,326


In [3]:
TARGETS = {
    'destination': 300,
    'plan_route': 150,
    'multi_turn': 60,
    'uncertain_dynamic': 50,
    'out_of_domain': 40,
}

approved = review_df[review_df['review_status'] == 'approved'].copy()

assert len(approved) == 600, f'Mora biti tačno 600 approved primjera, trenutno: {len(approved)}'
assert approved['final_category'].astype(str).str.strip().ne('').all(), 'Postoje approved primjeri bez final_category.'

actual = approved['final_category'].value_counts().to_dict()
print('Cilj:', TARGETS)
print('Trenutno:', actual)
assert actual == TARGETS, f'Raspodjela nije tačna. Trenutno: {actual}'

for category, target in TARGETS.items():
    print(f'{category:20s} {actual.get(category, 0):3d} / {target}')


Cilj: {'destination': 300, 'plan_route': 150, 'multi_turn': 60, 'uncertain_dynamic': 50, 'out_of_domain': 40}
Trenutno: {'destination': 300, 'plan_route': 150, 'multi_turn': 60, 'uncertain_dynamic': 50, 'out_of_domain': 40}
destination          300 / 300
plan_route           150 / 150
multi_turn            60 / 60
uncertain_dynamic     50 / 50
out_of_domain         40 / 40


In [4]:
# Provjera source_ids protiv stvarnog registra izvora.
assert SOURCES_PATH.exists(), f'Nedostaje: {SOURCES_PATH}'
sources_df = pd.read_csv(SOURCES_PATH, keep_default_na=False)
assert 'source_id' in sources_df.columns, 'sources.csv nema kolonu source_id.'

valid_source_ids = set(sources_df['source_id'].astype(str))
missing_source_ids = []

for _, row in approved.iterrows():
    source_ids = json.loads(row['source_ids'] or '[]')
    for sid in source_ids:
        if sid not in valid_source_ids:
            missing_source_ids.append((row['example_id'], sid))

assert not missing_source_ids, f'Nepostojeći source_ids: {missing_source_ids[:20]}'
print('source_ids provjera: OK')


source_ids provjera: OK


In [5]:
# Izgradi finalni skup koristeći PUNU messages_json strukturu.
final_examples = []

for _, row in approved.iterrows():
    messages = json.loads(row['messages_json'])
    destination_ids = json.loads(row['destination_ids'] or '[]')
    source_ids = json.loads(row['source_ids'] or '[]')

    assert isinstance(messages, list) and len(messages) >= 3, row['example_id']
    assert all(m.get('role') in {'system', 'user', 'assistant'} for m in messages), row['example_id']
    assert all(str(m.get('content', '')).strip() for m in messages), row['example_id']

    if row['final_category'] == 'multi_turn':
        roles = [m['role'] for m in messages]
        assert roles == ['system', 'user', 'assistant', 'user', 'assistant'], (row['example_id'], roles)

    final_examples.append({
        'example_id': row['example_id'],
        'type': row['type'],
        'destination_ids': destination_ids,
        'source_ids': source_ids,
        'reviewed': True,
        'messages': messages,
    })

assert len(final_examples) == 600
assert all(x['reviewed'] is True for x in final_examples)

multi_turn_count = sum(1 for _, row in approved.iterrows() if row['final_category'] == 'multi_turn')
assert multi_turn_count == 60

print('Finalna struktura poruka: OK')
print('Broj finalnih primjera:', len(final_examples))


Finalna struktura poruka: OK
Broj finalnih primjera: 600


In [6]:
# Sačuvaj all_examples.jsonl
with FINAL_PATH.open('w', encoding='utf-8') as f:
    for item in final_examples:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print('Sačuvan finalni skup:', FINAL_PATH)


Sačuvan finalni skup: /home/sinisa/Desktop/Sve/Fakultet/CETVRTA GODINA/DRUGI SEMESTAR/Vjestacka inteligencija/PROJEKAT/bih-tourist-chatbot/data/processed/all_examples.jsonl


In [7]:
# Završni T12 izvještaj
report_rows = []
for category, target in TARGETS.items():
    report_rows.append({
        'category': category,
        'target_count': target,
        'approved_count': actual.get(category, 0),
    })

report_rows.append({
    'category': 'TOTAL',
    'target_count': 600,
    'approved_count': len(approved),
})

report_df = pd.DataFrame(report_rows)
report_df.to_csv(REPORT_PATH, index=False, encoding='utf-8-sig')
display(report_df)
print('Sačuvan izvještaj:', REPORT_PATH)


,category,target_count,approved_count
0,destination,300,300
1,plan_route,150,150
2,multi_turn,60,60
3,uncertain_dynamic,50,50
4,out_of_domain,40,40
5,TOTAL,600,600


Sačuvan izvještaj: /home/sinisa/Desktop/Sve/Fakultet/CETVRTA GODINA/DRUGI SEMESTAR/Vjestacka inteligencija/PROJEKAT/bih-tourist-chatbot/artifacts/reports/sft_review_summary.csv


## T12 završna provjera

Task je završen kada sve ćelije prođu bez greške i postoji:
- `data/processed/all_examples.jsonl` sa tačno 600 primjera;
- raspodjela 300 / 150 / 60 / 50 / 40;
- `reviewed=true` za svaki finalni primjer;
- 60 stvarnih multi-turn primjera sa 5 poruka;
- nema praznih poruka;
- svi `source_ids` postoje u `data/sources.csv`.

Git commit:

`Review and finalize 600 training examples`
